# 06 · Generalized Hartree-Fock (GHF) with DIIS

GHF 把 alpha/beta 自旋自由度并入同一个 spin-orbital AO 基：

$$
\{ \chi_\mu \alpha, \chi_\mu \beta \}
$$

这样密度矩阵可以出现 alpha/beta 的 off-diagonal spin block。非相对论、无自旋轨道耦合的分子 Hamiltonian 本身仍然是 spin-free；如果从 block-diagonal 初猜开始，GHF 会退化到 UHF 解。

---

## 1 · Spin-Orbital HF 方程

用大写指标 $P,Q,R,S$ 表示 spin-orbital AO：

$$
\mathbf{F}\mathbf{C}
=
\mathbf{S}\mathbf{C}\boldsymbol{\varepsilon}
$$

其中

$$
F_{PQ}
=
H_{PQ}
+
\sum_{RS} D_{RS}
\left[
(PQ|RS) - (PR|QS)
\right]
$$

每个 spin orbital 的占据数为 0 或 1：

$$
D_{PQ} = \sum_i^{N_e} C_{Pi} C^*_{Qi}
$$

本 notebook 为了保持公式直接，显式构造 spin-orbital ERI。实际程序通常会利用 spin block 结构避免生成完整四维张量。

---

## 2 · 分子设定与 spin-orbital 积分

In [ ]:
from pyscf import gto
import numpy as np
from scipy.linalg import block_diag, eigh, fractional_matrix_power

# -------- 分子定义: OH radical / cc-pVDZ --------
mol = gto.M(
    atom="O 0 0 0; H 0 0 0.9697",
    basis="cc-pvdz",
    spin=1,
    verbose=0,
)
nao = mol.nao_nr()
nso = 2 * nao
nelec = mol.nelectron
nalpha, nbeta = mol.nelec

# -------- 空间 AO 积分 --------
S = mol.intor("int1e_ovlp_sph")
T = mol.intor("int1e_kin_sph")
V = mol.intor("int1e_nuc_sph")
H_core = T + V
A = fractional_matrix_power(S, -0.5)
eri = mol.intor("int2e")
E_nn = mol.energy_nuc()

# -------- spin-orbital AO 矩阵 --------
S_so = block_diag(S, S)
H_core_so = block_diag(H_core, H_core)
A_so = block_diag(A, A)


def build_spinorb_eri(eri):
    nao = eri.shape[0]
    nso = 2 * nao
    eri_so = np.zeros((nso, nso, nso, nso), dtype=eri.dtype)
    spin_blocks = (slice(0, nao), slice(nao, nso))

    # (mu sigma, nu sigma | kappa tau, lambda tau)
    for same_spin_1 in spin_blocks:
        for same_spin_2 in spin_blocks:
            eri_so[same_spin_1, same_spin_1, same_spin_2, same_spin_2] = eri
    return eri_so


eri_so = build_spinorb_eri(eri)

print(f"nao = {nao}, nso = {nso}, nelec = {nelec}")
print(f"nalpha = {nalpha}, nbeta = {nbeta}")
print(f"S_so shape = {S_so.shape}, H_core_so shape = {H_core_so.shape}")
print(f"eri_so shape = {eri_so.shape} ({eri_so.size:,} elements)")
print(f"E_nn = {E_nn:.10f} Hartree")

---

## 3 · SCF 核心函数

In [ ]:
# ============================================================
# 3.1 占据数: 最低 nelec 个 spin orbital 各占 1 个电子
# ============================================================
def get_occ(mo_energy, nocc):
    e_idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    mo_occ[e_idx[:nocc]] = 1
    return mo_occ


# ============================================================
# 3.2 密度矩阵: D = C_occ C_occ^dagger
# ============================================================
def make_rdm1(mo_coeff, mo_occ):
    mocc = mo_coeff[:, mo_occ > 0]
    occ = mo_occ[mo_occ > 0]
    return (mocc * occ).dot(mocc.conj().T)


# ============================================================
# 3.3 J/K 与有效势
# J_{PQ} = sum_{RS} D_{RS} (PQ|RS)
# K_{PQ} = sum_{RS} D_{RS} (PR|QS)
# ============================================================
def get_jk(eri_so, dm):
    vj = np.einsum("pqrs,rs->pq", eri_so, dm, optimize=True)
    vk = np.einsum("prqs,rs->pq", eri_so, dm, optimize=True)
    return vj, vk


def get_veff(eri_so, dm):
    vj, vk = get_jk(eri_so, dm)
    return vj - vk


def get_fock(H_core_so, V_eff):
    return H_core_so + V_eff


# ============================================================
# 3.4 电子能量
# ============================================================
def energy_elec(dm, H_core_so, V_eff):
    e1 = np.einsum("pq,qp->", H_core_so, dm, optimize=True)
    e2 = 0.5 * np.einsum("pq,qp->", V_eff, dm, optimize=True)
    return (e1 + e2).real, e2.real


def energy_tot(dm, H_core_so, V_eff):
    return E_nn + energy_elec(dm, H_core_so, V_eff)[0]


# ============================================================
# 3.5 block-diagonal UHF-like 初猜
# ============================================================
def get_init_guess(H_core, S):
    mo_energy, mo_coeff = eigh(H_core, S)

    occ_alpha = np.zeros_like(mo_energy)
    occ_beta = np.zeros_like(mo_energy)
    e_idx = np.argsort(mo_energy)
    occ_alpha[e_idx[:nalpha]] = 1
    occ_beta[e_idx[:nbeta]] = 1

    dm_alpha = make_rdm1(mo_coeff, occ_alpha)
    dm_beta = make_rdm1(mo_coeff, occ_beta)
    return block_diag(dm_alpha, dm_beta)


print("GHF helper functions ready.")

---

## 4 · DIIS

In [ ]:
def get_err_vec(fock, dm, S_so, A_so):
    return A_so @ (fock @ dm @ S_so - S_so @ dm @ fock) @ A_so


def apply_diis(fock_list, err_vec_list):
    n = len(fock_list)
    dim = n + 1
    B = np.empty((dim, dim))
    B[-1, :] = -1
    B[:, -1] = -1
    B[-1, -1] = 0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.vdot(err_vec_list[i], err_vec_list[j]).real

    rhs = np.zeros(dim)
    rhs[-1] = -1

    try:
        coeff = np.linalg.solve(B, rhs)[:-1]
    except np.linalg.LinAlgError:
        coeff = np.linalg.lstsq(B, rhs, rcond=None)[0][:-1]

    return np.einsum("i,ipq->pq", coeff, np.asarray(fock_list), optimize=True)


print("DIIS functions defined.")

---

## 5 · GHF + DIIS 完整运行

In [ ]:
max_cycle = 80
max_diis = 8
conv_tol = 1e-10
conv_tol_dm = 1e-8

e_old = 0.0
fock_list = []
err_vec_list = []

dm = get_init_guess(H_core, S)

print(f"{'Iter':>4s}  {'E_total':>16s}  {'dE':>10s}  {'dD':>10s}")
print("-" * 46)

converged = False
for cycle in range(max_cycle):
    V_eff = get_veff(eri_so, dm)
    fock = get_fock(H_core_so, V_eff)
    e_tot = energy_tot(dm, H_core_so, V_eff)

    err = get_err_vec(fock, dm, S_so, A_so)
    fock_list.append(fock)
    err_vec_list.append(err)
    if len(fock_list) > max_diis:
        fock_list.pop(0)
        err_vec_list.pop(0)

    if cycle >= 2:
        fock = apply_diis(fock_list, err_vec_list)

    mo_energy, mo_coeff = eigh(fock, S_so)
    mo_occ = get_occ(mo_energy, nelec)
    dm_new = make_rdm1(mo_coeff, mo_occ)

    e_diff = abs(e_tot - e_old)
    d_norm = np.linalg.norm(dm_new - dm)

    print(f"{cycle+1:4d}  {e_tot:16.10f}  {e_diff:10.3e}  {d_norm:10.3e}")

    if e_diff < conv_tol and d_norm < conv_tol_dm:
        converged = True
        print(f"\nConverged in {cycle+1} iterations!")
        print(f"E_total (GHF+DIIS/cc-pVDZ) = {e_tot:.10f} Hartree")
        print(f"E_elec = {e_tot - E_nn:.10f}")
        print(f"E_nn   = {E_nn:.10f}")
        break

    dm = dm_new
    e_old = e_tot

if not converged:
    print("WARNING: GHF SCF did not converge!")

---

## 6 · 与 PySCF 对照

In [ ]:
from pyscf import scf

mf_ref = scf.GHF(mol)
mf_ref.conv_tol = conv_tol
mf_ref.kernel(dm0=dm)

spin_flip_norm = np.linalg.norm(dm[:nao, nao:]) + np.linalg.norm(dm[nao:, :nao])

print(f"\nPySCF GHF:     {mf_ref.e_tot:.10f} Hartree")
print(f"Our GHF+DIIS:   {e_tot:.10f} Hartree")
print(f"Difference:     {abs(mf_ref.e_tot - e_tot):.2e} Hartree")
print(f"||D_alpha_beta|| + ||D_beta_alpha|| = {spin_flip_norm:.3e}")

---

## 7 · 总结

| 模块 | GHF 形式 |
|------|----------|
| 基 | $\chi_\mu\alpha$ 与 $\chi_\mu\beta$ 合并成 spin-orbital AO |
| 密度 | 一个 $2N_\mathrm{AO}\times 2N_\mathrm{AO}$ 的总密度矩阵 |
| Fock | $F_{PQ}=H_{PQ}+\sum_{RS}D_{RS}[(PQ|RS)-(PR|QS)]$ |
| 占据 | 最低 $N_e$ 个 spin orbital 各占 1 个电子 |
| 与 UHF 关系 | 无 spin mixing 且密度 block diagonal 时，GHF 退化为 UHF |